# Simple 1D Monte Carlo Integration

In [9]:
import numpy as np
import scipy.integrate as integrate

b = 3
a = 0

N = 10000

x_input = np.linspace(a, b, N)
y = x_input ** 3 - x_input ** 2
integral_result = integrate.simpson(y, x=x_input)

x_random = np.random.uniform(a, b, N)

integral_result_monte_carlo = (b - a) / N * np.sum(x_random ** 3 - x_random ** 2)

print(integral_result_monte_carlo),
print(integral_result)


11.234971414055922
11.250000000000002


# Harmonic Oscillator

In [1]:
import numpy as np


def psi_test(alpha, x):
    return np.exp(-alpha * x ** 2)


def local_energy(alpha, x):
    return alpha + x ** 2 * (-2 * alpha ** 2 + 1 / 2)


def vmc_simulation(N, alpha, walker_steps, ignored_walkers, d):
    walker_positions = np.random.uniform(-1, 1, N)
    walker_energies_sum = np.zeros(N)
    walker_energies_squared_sum = np.zeros(N)

    for step in range(walker_steps):
        proposed_new_position = walker_positions + np.random.uniform(-d / 2, +d / 2, N)
        probability = (psi_test(alpha, proposed_new_position) / psi_test(alpha, walker_positions)) ** 2
        will_move = np.random.random(N) < probability
        walker_positions[will_move] = proposed_new_position[will_move]
        if step >= ignored_walkers:
            e_l = local_energy(alpha, walker_positions)
            walker_energies_sum += e_l
            walker_energies_squared_sum += e_l ** 2

    e_l_expected = np.mean(walker_energies_sum / (walker_steps - ignored_walkers))
    e_l_squared_expected = np.mean(walker_energies_squared_sum) / (walker_steps - ignored_walkers)
    return e_l_expected, e_l_squared_expected - e_l_expected ** 2

In [2]:
import pandas as pd

alpha_values = [0.4, 0.45, 0.5, 0.55, 0.6]
results = []
for alpha_trial in alpha_values:
    expected_energy, expected_variance = vmc_simulation(N=400, alpha=alpha_trial, walker_steps=30000,
                                                        ignored_walkers=4000, d=0.8)
    results.append({"Alpha": alpha_trial, "<E>": expected_energy, "<V>": expected_variance})

df = pd.DataFrame(results)
df

,Alpha,<E>,<V>
0,0.40,0.512550,0.025254
1,0.45,0.502757,0.005567
2,0.50,0.500000,0.000000
3,0.55,0.502164,0.004580
4,0.60,0.508335,0.016792


In [50]:
import line_profiler

lp = line_profiler.LineProfiler(vmc_simulation)
lp.runcall(vmc_simulation, N=400, alpha=0.4, walker_steps=30000, ignored_walkers=4000, d=0.8)
lp.print_stats()

Timer unit: 1e-09 s

Total time: 0.342563 s
File: /var/folders/jk/vyz1f6h92v17hl9dr4l3c1vc0000gn/T/ipykernel_5312/635450587.py
Function: vmc_simulation at line 9

Line #      Hits         Time  Per Hit   % Time  Line Contents
     9                                           def vmc_simulation(N, alpha, walker_steps, ignored_walkers, d):
    10         1      42000.0  42000.0      0.0      walker_positions = np.random.uniform(-1,1, N)
    11         1       5000.0   5000.0      0.0      walker_energies_sum = np.zeros(N)
    12                                           
    13     30001    4277000.0    142.6      1.2      for step in range(walker_steps):
    14     30000   70811000.0   2360.4     20.7          proposed_new_position = walker_positions + np.random.uniform(-d / 2, +d / 2, N)
    15     30000  130271000.0   4342.4     38.0          probability = (psi_test(alpha, proposed_new_position) / psi_test(alpha, walker_positions)) ** 2
    16     30000   42883000.0   1429.4     12.5  

In [36]:
results = []

for i in range(10):
    results.append(vmc_simulation(N=400, alpha=0.4, walker_steps=30000, ignored_walkers=4000, d=0.8))